In [29]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage
from IPython.display import Markdown

In [30]:
load_dotenv(override=True)

True

In [31]:
mcp_client = MultiServerMCPClient(
    {
        "mcp_server": {
            "transport": "streamable_http",
            "url": "http://localhost:24000/mcp",
        }
    }
)

In [32]:
tools = await mcp_client.get_tools()

In [33]:
tools

[StructuredTool(name='get_employee_infos', description='"Get info about the given employee', args_schema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_employee_infosArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000023F114C7880>),
 StructuredTool(name='search_web', description='"Search web for the given query', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_webArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000023F15F41E40>)]

In [34]:
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)

In [35]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="answer user question using provided tools"
)

In [36]:
resp = await agent.ainvoke({
    "messages": [
        HumanMessage("Quel est le salaire de Hassan")
    ]
})

In [37]:
print(resp['messages'][-1].content)

Le salaire de Hassan est de 23 000.


In [38]:
resp = await agent.ainvoke({
    "messages": [
        HumanMessage("Quelle était la valeur de l'action de Microsoft la semaine dernière ?")    ]
})

In [40]:
print(display(Markdown(resp['messages'][-1].content)))

La valeur de l'action de Microsoft la semaine dernière a varié. Voici quelques détails :

- Le cours de l'action a atteint un minimum de **415,75 $** et un maximum de **429,92 $** au cours de la semaine.
- À la clôture, le prix était de **429,25 $**.

Pour plus d'informations, vous pouvez consulter [ce lien](https://www.boursedirect.fr/fr/marche/nasdaq-ngs-global-select-market/microsoft-corporation-MSFT-USD-XNGS/seance).

None
